# Testing the Capacity-Magnitude Hypothesis

## The hypothesis, stated precisely and falsifiably

Per MODEL_RESULTS_LOG.md: the three faults that degrade forward-in-time (undercharge,
evaporator fouling, suction-line restriction) all have large-magnitude capacity swings
as a major effect. The three stable faults do not. Hypothesis: capacity's own
substantial weather-driven variance interacts poorly with a large fault effect on
capacity specifically, degrading forward-in-time generalization.

## The test

For each of the 3 degrading faults, refit the SAME model with capacity REMOVED from
the feature set, keeping everything else identical (same pressure/temperature
features, same weather-residualization, same TimeSeriesSplit evaluation). If the
hypothesis is correct, removing capacity should substantially reduce or eliminate
the degradation trend, since the problematic feature is gone - even if overall
performance drops somewhat (capacity was carrying real signal, per the EDA).

If degradation persists even without capacity, the hypothesis is falsified, and
something else is driving the pattern - genuinely useful to know either way.

In [1]:
import sys
from pathlib import Path

ml_root = Path.cwd().parent
if str(ml_root) not in sys.path:
    sys.path.insert(0, str(ml_root))

from sklearn.ensemble import RandomForestClassifier  # noqa: E402
from sklearn.metrics import classification_report  # noqa: E402
from sklearn.model_selection import TimeSeriesSplit  # noqa: E402
from src.features.build_features import build_feature_table  # noqa: E402


def evaluate_without_capacity(table, feature_cols_no_capacity, fault_label):
    """Run TimeSeriesSplit evaluation using only non-capacity features."""
    X_all = table[feature_cols_no_capacity].values
    y_all = table["label"].values
    tscv = TimeSeriesSplit(n_splits=5)

    print(f"=== {fault_label}: TimeSeriesSplit WITHOUT capacity ===")
    for fold_num, (train_idx, test_idx) in enumerate(tscv.split(X_all), start=1):
        X_tr, X_te = X_all[train_idx], X_all[test_idx]
        y_tr, y_te = y_all[train_idx], y_all[test_idx]
        model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_te)
        report = classification_report(y_te, y_pred, target_names=["baseline", fault_label], output_dict=True)
        print(f"Fold {fold_num}: baseline recall={report['baseline']['recall']:.2f}, "
              f"baseline precision={report['baseline']['precision']:.2f}")


# --- Undercharge, without capacity ---
table_uc = build_feature_table(
    baseline_path="../data/raw/RTU_sim_baseline.csv",
    fault_paths={
        "undercharge10": "../data/raw/RTU_sim_undercharge10.csv",
        "undercharge15": "../data/raw/RTU_sim_undercharge15.csv",
        "undercharge20": "../data/raw/RTU_sim_undercharge20.csv",
    },
)
evaluate_without_capacity(
    table_uc,
    ["RTU_REFG_SUCT_PRES_residual", "RTU_REFG_SUCT_TEMP_residual", "RTU_REFG_DISC_PRES_residual"],
    "undercharge",
)

=== undercharge: TimeSeriesSplit WITHOUT capacity ===
Fold 1: baseline recall=0.12, baseline precision=0.92
Fold 2: baseline recall=0.07, baseline precision=0.95
Fold 3: baseline recall=0.02, baseline precision=0.95
Fold 4: baseline recall=0.01, baseline precision=0.70
Fold 5: baseline recall=0.02, baseline precision=0.79


## Result: hypothesis FALSIFIED for undercharge — removing capacity made things worse,
## not better

| | With capacity (notebook 11) | Without capacity |
|---|---|---|
| Fold 1 | 0.44 | 0.12 |
| Fold 2 | 0.24 | 0.07 |
| Fold 3 | 0.06 | 0.02 |
| Fold 4 | 0.02 | 0.01 |
| Fold 5 | 0.00 | 0.02 |

Removing capacity did NOT fix undercharge's degradation - it made every fold worse.
This directly falsifies the capacity-magnitude hypothesis as stated: if large
capacity swings interacting poorly with weather variance were the cause, removing
capacity should have improved things, not degraded them further. Instead, this
suggests capacity was actually the MOST HELPFUL feature undercharge had (removing it
hurts more than keeping it), and the real problem lies with the remaining pressure/
temperature features - or with something else about undercharge specifically, not a
general "capacity is bad" property shared across faults.

**This is a real, valuable negative result** - it prevents drawing a false general
conclusion from a correlational pattern across 6 data points. The three-way split
(stable/gradual-decline/collapse) is real and worth documenting, but the proposed
mechanism (capacity-driven) does not hold up under direct testing.

In [2]:
# --- Evaporator fouling, without capacity ---
table_ef = build_feature_table(
    baseline_path="../data/raw/RTU_sim_baseline.csv",
    fault_paths={
        "evapfouling10": "../data/raw/RTU_sim_evapfouling10.csv",
        "evapfouling20": "../data/raw/RTU_sim_evapfouling20.csv",
        "evapfouling30": "../data/raw/RTU_sim_evapfouling30.csv",
        "evapfouling40": "../data/raw/RTU_sim_evapfouling40.csv",
        "evapfouling50": "../data/raw/RTU_sim_evapfouling50.csv",
    },
    pressure_temp_cols=("RTU_REFG_SUCT_PRES", "RTU_REFG_SUCT_TEMP", "RTU_SA_TEMP"),
)
evaluate_without_capacity(
    table_ef,
    ["RTU_REFG_SUCT_PRES_residual", "RTU_REFG_SUCT_TEMP_residual", "RTU_SA_TEMP_residual"],
    "evapfouling",
)

# --- Suction-line restriction, without capacity ---
table_sl = build_feature_table(
    baseline_path="../data/raw/RTU_sim_baseline.csv",
    fault_paths={
        "suctionpipe01bar": "../data/raw/RTU_sim_suctionpipe01bar.csv",
        "suctionpipe03bar": "../data/raw/RTU_sim_suctionpipe03bar.csv",
        "suctionpipe06bar": "../data/raw/RTU_sim_suctionpipe06bar.csv",
        "suctionpipe09bar": "../data/raw/RTU_sim_suctionpipe09bar.csv",
    },
    pressure_temp_cols=("RTU_REFG_SUCT_PRES", "RTU_REFG_SUCT_TEMP"),
)
evaluate_without_capacity(
    table_sl,
    ["RTU_REFG_SUCT_PRES_residual", "RTU_REFG_SUCT_TEMP_residual"],
    "suctionline",
)

=== evapfouling: TimeSeriesSplit WITHOUT capacity ===
Fold 1: baseline recall=0.70, baseline precision=0.72
Fold 2: baseline recall=0.73, baseline precision=0.73
Fold 3: baseline recall=0.77, baseline precision=0.72
Fold 4: baseline recall=0.79, baseline precision=0.73
Fold 5: baseline recall=0.82, baseline precision=0.74
=== suctionline: TimeSeriesSplit WITHOUT capacity ===
Fold 1: baseline recall=0.98, baseline precision=0.63
Fold 2: baseline recall=0.99, baseline precision=0.63
Fold 3: baseline recall=0.99, baseline precision=0.63
Fold 4: baseline recall=0.98, baseline precision=0.63
Fold 5: baseline recall=0.96, baseline precision=0.63


## Result: hypothesis CONFIRMED for evaporator fouling and suction-line restriction —
## but FALSIFIED for undercharge. A more precise, nuanced finding, not a uniform rule.

**Evaporator fouling:**
| | With capacity | Without capacity |
|---|---|---|
| Fold 1 | 0.76 | 0.70 |
| Fold 2 | 0.60 | 0.73 |
| Fold 3 | 0.53 | 0.77 |
| Fold 4 | 0.45 | 0.79 |
| Fold 5 | 0.41 | 0.82 |

**The degradation trend is not just reduced - it REVERSES.** Recall now IMPROVES
across folds (0.70→0.82) instead of declining (0.76→0.41). Precision drops
(0.93-0.97 -> 0.72-0.74 - real cost) but the forward-in-time instability is gone.

**Suction-line restriction:**
| | With capacity | Without capacity |
|---|---|---|
| Fold 1 | 0.87 | 0.98 |
| Fold 2 | 0.77 | 0.99 |
| Fold 3 | 0.68 | 0.99 |
| Fold 4 | 0.60 | 0.98 |
| Fold 5 | 0.43 | 0.96 |

**Even more dramatic** - recall is now stable and near-perfect throughout (0.96-0.99)
instead of collapsing (0.87→0.43). Precision drops from ~1.00 to a consistent 0.63 -
a real, meaningful cost, but a completely different, stable failure mode (consistent
false-alarm rate) rather than an unpredictable, worsening-over-time one.

**Revised conclusion: the capacity-magnitude hypothesis is TRUE for 2 of 3 faults,
false for undercharge specifically.** This is a more precise and more useful finding
than either "capacity always causes this" or "capacity is unrelated" - it means
undercharge has a genuinely different, still-unexplained cause for its
forward-in-time instability, while evaporator fouling and suction-line restriction's
instability IS explained by capacity's interaction with weather variance.

**Real, practical implication for the modeling phase**: for evaporator fouling and
suction-line restriction, dropping capacity trades some precision for much more
reliable, stable recall over time - likely the better tradeoff for a real deployed
system where consistent behavior matters more than squeezing out maximum accuracy.
For undercharge, this fix doesn't apply - it needs its own dedicated investigation,
separate from the capacity theory, before it can be considered resolved.

## Summary: capacity-ablation test — a nuanced, real finding, not a uniform rule

Tested the capacity-magnitude hypothesis directly across all 3 degrading faults by
removing capacity from the feature set:

- **Evaporator fouling**: CONFIRMED — degradation trend reverses to improvement
  (0.76→0.41 becomes 0.70→0.82), at a real precision cost (0.93-0.97 → 0.72-0.74).
- **Suction-line restriction**: CONFIRMED, even more dramatically — recall becomes
  stable and near-perfect (0.87→0.43 becomes 0.96-0.99 stable), at a real precision
  cost (~1.00 → consistent 0.63).
- **Undercharge**: FALSIFIED — removing capacity made every fold worse, not better.
  Capacity appears to be undercharge's most helpful feature, not its problem.

**Practical takeaway**: for evaporator fouling and suction-line restriction, use the
capacity-free feature set for real deployment — trades some precision for far more
reliable, predictable behavior over time. For undercharge, this fix does not apply;
its forward-in-time instability remains genuinely unresolved and needs separate
investigation, not assumed to share evaporator fouling/suction-line restriction's
cause just because all three showed similar symptoms.